# 🧠 Entrenamiento del MLP — Oil Slick Classifier
**TAMI 2026 · YPF**

Entrena el `OilSlickMLP` sobre el dataset de Kaggle y guarda los artefactos
necesarios para el pipeline de producción.

> **Prerequisito:** correr `descarga_kaggle.ipynb` para tener el dataset en `data/kaggle_dataset/`.

**Artefactos generados:**
- `models/feature_net_model.pth` — pesos del modelo
- `models/scaler.joblib` — StandardScaler ajustado al dataset de entrenamiento

**Métricas obtenidas (75/25 split):**

| Métrica | KNN | MLP |
|---------|-----|-----|
| Precisión | 84.0% | **85.0%** |
| Recall | 64.0% | **75.4%** |
| F1-Score | 73.0% | **80.0%** |
| Accuracy | 83.8% | **87.2%** |

## 0. Setup

In [ ]:
# ── Colab ────────────────────────────────────────────────────────────────────
from google.colab import drive
import sys

drive.mount('/content/drive')

REPO_PATH = '/content/drive/MyDrive/oil-slick-detection'
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q scikit-image scikit-learn torch joblib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.neighbors import KNeighborsClassifier

from src.features import FEATURE_NAMES
from src.config import PATHS

print('Setup completo.')

## 1. Construcción del dataset de features

Corre el pipeline de preprocesamiento sobre cada imagen del dataset de Kaggle
y extrae el vector de 16 características.

> Si ya tenés el CSV pre-computado (`data/features_dataset.csv`), salteá esta celda
> y cargalo directamente en la celda **1b**.

In [ ]:
# --- 1a. Extraer features ---
import cv2 as cv
from src.preprocessing import lee_filter, apply_mixed_filter
from src.features import extract_features

DATASET_DIR  = Path(REPO_PATH) / 'data' / 'kaggle_dataset'
FEATURES_CSV = Path(REPO_PATH) / 'data' / 'features_dataset.csv'

records = []

for label_str in ['Oil', 'No_Oil']:
    class_dir = DATASET_DIR / label_str
    if not class_dir.exists():
        print(f'Carpeta no encontrada: {class_dir}')
        continue

    imgs = sorted(list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')))
    print(f'{label_str}: {len(imgs)} imágenes')

    for img_path in imgs:
        img = cv.imread(str(img_path), cv.IMREAD_GRAYSCALE)
        if img is None:
            continue
        filtered = lee_filter(img.astype(np.float32))
        mask     = apply_mixed_filter(filtered.astype(np.int32))
        feats    = extract_features(filtered.astype(np.int32), mask)
        if feats is not None:
            feats['label'] = label_str
            feats['image'] = img_path.name
            records.append(feats)

df = pd.DataFrame(records)
df.to_csv(FEATURES_CSV, index=False)
print(f'\n{len(df)} muestras extraidas. Guardado en {FEATURES_CSV}')
print(df['label'].value_counts())

In [ ]:
# --- 1b. (Alternativa) Cargar CSV pre-computado ---
# FEATURES_CSV = Path(REPO_PATH) / 'data' / 'features_dataset.csv'
# df = pd.read_csv(FEATURES_CSV)

print('Dataset shape:', df.shape)
print(df.head())

## 2. Split features y labels

In [ ]:
X = df.drop(columns=['label', 'image']).values
y = df['label'].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print('Clases:', label_encoder.classes_)   # ['No_Oil' 'Oil'] -> [0, 1]
print('Distribución:', dict(zip(*np.unique(y, return_counts=True))))

## 3. Train / Test split (75/25, estratificado)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras')

## 4. Normalización de features

In [ ]:
# Ajustar SOLO sobre train para evitar data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Convertir a tensores
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=16,
    shuffle=True
)

## 5. Definición de la red neuronal

```
Input (16) → Linear(32) → ReLU → Linear(16) → ReLU → Linear(2)
```

In [ ]:
from src.model import OilSlickMLP

input_size  = X_train_t.shape[1]           # 16
num_classes = len(label_encoder.classes_)  # 2

model = OilSlickMLP(input_dim=input_size, num_classes=num_classes)
print(model)

## 6. Entrenamiento

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f'Epoch {epoch} Loss {total_loss:.4f}')

## 7. Evaluación

In [ ]:
model.eval()

with torch.no_grad():

    outputs = model(X_test_t)

    _, predictions = torch.max(outputs, 1)

y_pred = predictions.numpy()
y_true = y_test_t.numpy()

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print('\nOverall Accuracy:', accuracy)

# Classification report
print('\nClassification Report:\n')
print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print('\nConfusion Matrix:\n')
print(cm)

# Accuracy per class
print('\nAccuracy per class:')
for i, cls in enumerate(label_encoder.classes_):
    correct = cm[i, i]
    total = cm[i].sum()
    acc = correct / total if total > 0 else 0
    print(f'{cls}: {acc:.4f}')

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    ax=ax
)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Confusion Matrix — OilSlickMLP')
plt.tight_layout()
plt.show()

## 8. Baseline: K-Nearest Neighbors

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print('=== KNN ===')
print(classification_report(
    y_test, y_pred_knn,
    target_names=label_encoder.classes_
))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_knn))

## 9. Guardar artefactos del modelo

In [ ]:
PATHS['models_dir'].mkdir(parents=True, exist_ok=True)

weights_path = PATHS['models_dir'] / 'feature_net_model.pth'
torch.save(model.state_dict(), weights_path)

scaler_path = PATHS['models_dir'] / 'scaler.joblib'
joblib.dump(scaler, scaler_path)

print(f'Pesos guardados en:  {weights_path}')
print(f'Scaler guardado en:  {scaler_path}')
print(f'\nClases: {label_encoder.classes_.tolist()}')
print('  -> Clase 0 = No_Oil  |  Clase 1 = Oil')